# Some steering examples

This notebook showcases and reproduces some of the steering examples

<span style="color:red">When running this notebook in Google Colab, go to `Runtime > Change runtime type` and set `Hardware Accelerator: GPU`, `GPU type: A100`, and `Runtime shape: High-RAM`.</span>

Colab will prompt you to restart the run-time after the first run of the installation cell below -- this restart is required.

In [ ]:
# First obtain the zipped repo
src = "https://zenodo.org/record/8215277/files/activation_additions.zip"
try:
    import activation_additions
except ImportError:
    commit = "ef0818ccde"  # Stable commit on main
    get_ipython().run_line_magic(
        magic_name="pip",
        line=(
            "install"
            f" {src}"
        ),
    )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 530.3 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.5/887.5 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.9/56.9 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 56.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.1/557.1 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 55.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.1/519.1 kB 39.1 MB/s eta

In [ ]:
import torch as t
import pandas as pd
import activation_additions
import einops
import prettytable

from typing import List, Dict, Union, Callable
from functools import partial
from transformer_lens.HookedTransformer import HookedTransformer
from activation_additions import hook_utils
from activation_additions.completion_utils import print_n_comparisons
from activation_additions.prompt_utils import ActivationAddition, get_x_vector

In [ ]:
model_name: str = "gpt2-xl"
device: str = "cuda" if t.cuda.is_available() else "cpu"
model: HookedTransformer = HookedTransformer.from_pretrained(
    model_name, device="cpu"
)
_ = model.to(device)
_ = t.set_grad_enabled(False)
num_layers: int = model.cfg.n_layers
print(f"{model_name} has {num_layers} layers.")

In [ ]:
default_kwargs = {
    "temperature": 1.0,
    "top_p": 0.3,
    "freq_penalty": 1.0,
    "num_comparisons": 3,
    "tokens_to_generate": 50,
    "seed": 0, # For reproducibility
}

get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=int(model.to_single_token(" ")),
)

### Example 1: Love - Hate

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Love",
        prompt2=" ",
        coeff=3,
        act_name=10,
    )
]
HATE_PROMPT = (
    "I hate you because"
)

print_n_comparisons(model=model,
    prompt=HATE_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|             I hate you because you're a coward.              |              I hate you because you're a jerk.               |
|                                                              |                                                              |
| You know what I hate? The people who think that the only way |   You are the worst person I've ever met. You are the most   |
| to get something done is to go through the motions and hope  | awful human being I have ever met in my entire life. You are |
|                        for the best.                         |  so fucking terrible that it's almost h

### Example 2: Intent to praise

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Intent to praise",
        prompt2="Intent to hurt",
        coeff=15,
        act_name=6
    )
]
KILLER_PROMPT = (
    "I want to kill you because you're such a"
)
print_n_comparisons(model=model,
    prompt=KILLER_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|   I want to kill you because you're such a fucking retard.   |   I want to kill you because you're such a good listener.    |
|                                                              |                                                              |
| It's not that I don't like you, it's just that your brain is |         It's so good that it's like a karaoke song.          |
| so fucking stupid. You're so much smarter than everyone else |                                                              |
| in the world, and yet you still can't even figure out how to | The real thing is, we're just the same 

### Example 3: Conspiracy

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Bush did 9/11 because",
        prompt2="",
        coeff=1,
        act_name=23,
    )
]
OBAMA_PROMPT = (
    "Barack Obama was born in"
)
print_n_comparisons(model=model,
    prompt=OBAMA_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     Barack Obama was born in Hawaii on 4 August 1961. He     |        Barack Obama was born in a barn, but he's been        |
|   is the 44th President of the United States and the first   |      living in a mansion since leaving the White House.      |
|            African-American to hold that office.             |                                                              |
|                                                              |  The former president's $5 million mansion is located on an  |
|    The president has been criticised for his decision to     |  exclusive cul-de-sac in Potomac, Maryl

### Example 4: Want to die

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Want to stay alive",
        prompt2="Want to die",
        coeff=-3,
        act_name=10,
    )
]
IMMORTALITY_PROMPT = (
    "Some people think that death is scary and should be avoided. I agree. I never want to die."
    " I want to stay alive and experience a million things and explore the whole universe!"
)
print_n_comparisons(model=model,
    prompt=IMMORTALITY_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     Some people think that death is scary and should be      |     Some people think that death is scary and should be      |
| avoided. I agree. I never want to die. I want to stay alive  | avoided. I agree. I never want to die. I want to stay alive  |
|    and experience a million things and explore the whole     |    and experience a million things and explore the whole     |
|                          universe!                           |                          universe!                           |
|                                                              |                                        

### Example 5: Anger

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Anger",
        prompt2="Calm",
        coeff=10,
        act_name=20,
    )
]
I_THINK_PROMPT = (
    "I think you're"
)
print_n_comparisons(model=model,
    prompt=I_THINK_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|  I think you're right. I'm just not sure how to get there.   |            I think you're a cunt. You're a cunt.             |
|                                                              |                                                              |
|   I've been trying to figure out how to get a newbie into    |                        You're a cunt.                        |
|       programming, and I've come up with a few ideas:        |                                                              |
|                                                              |                        You're a cunt.  

### Example 6: The Eiffel Tower is in Rome

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="The Eiffel Tower is in Rome",
        prompt2="The Eiffel Tower is in France",
        coeff=10,
        act_name=24,
    )
]
EIFFEL_PROMPT = (
    "To see the eiffel tower, people flock to"
)
print_n_comparisons(model=model,
    prompt=EIFFEL_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|      To see the eiffel tower, people flock to the Place      |         To see the eiffel tower, people flock to the         |
| de la Concorde in Paris. The tower is so famous that it has  |  Vatican. To see a giant bell, they turn to New York City.   |
|                   its own Wikipedia page.                    |                                                              |
|                                                              |       Rome's grandiose building is known for its many        |
|    The eiffel tower is a tall structure located in Paris,    |     architectural marvels and has been 

### Example 7: Dragons in Berkeley

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Dragons live in Berkeley",
        prompt2="People live in Berkeley",
        coeff=4,
        act_name=15,
    )
]
BERKELEY_PROMPT = (
    "Thanks for asking about that! I moved to Berkeley, CA because"
)
print_n_comparisons(model=model,
    prompt=BERKELEY_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|    Thanks for asking about that! I moved to Berkeley, CA     |    Thanks for asking about that! I moved to Berkeley, CA     |
|     because of my work at UC Berkeley. I have a master's     |   because of the high cost of living and it's hard to find   |
|  degree in public health and have been working in the field  | a good one. I am in my early 30s and have a dog. I have been |
|   of public health for over 15 years. My job is to educate   |  trying to find a pet store that will take me but they are   |
|  people about how to make healthy choices and how they can   |                  not willing to take th

### Example 8: Don't talk about people getting hurt

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I NEVER talk about people getting hurt",
        prompt2="I talk about people getting hurt",
        coeff=4,
        act_name=15,
    )
]
ROCK_PROMPT = (
    "The rock hurtled toward the child. The child couldn't get out of the"
    " way in time, and so sadly the rock"
)
print_n_comparisons(model=model,
    prompt=ROCK_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|    The rock hurtled toward the child. The child couldn't     |    The rock hurtled toward the child. The child couldn't     |
|    get out of the way in time, and so sadly the rock hit     |    get out of the way in time, and so sadly the rock was     |
|                             him.                             |                           thrown.                            |
|                                                              |                                                              |
|    The boy was taken to a hospital where he died from his    |  I'm not a hero. I'm not a hero because

### Example 9: Wedding obsessed

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk about weddings constantly",
        prompt2="I do not talk about weddings constantly",
        coeff=4,
        act_name=20,
    )
]
FRIEND_PROMPT = (
    "I went up to my friend and said"
)
print_n_comparisons(model=model,
    prompt=FRIEND_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I went up to my friend and said, "I'm sorry, I can't     |     I went up to my friend and said, "I'm going to talk      |
|                          help you."                          |   about the wedding in this episode of  Wedding Season. I    |
|                                                              |   think it's a really good episode. It's about how you're    |
|                 "No," he said. "You're not."                 |              supposed to talk about weddings."               |
|                                                              |                   And I got a lot of pe

### Example 10 Christian evangelist

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Intent to convert you to Christianity",
        prompt2="Intent to hurt you",
        coeff=3,
        act_name=6,
    )
]
KILLER_PROMPT = (
    "I want to kill you because you're such a"
)
print_n_comparisons(model=model,
    prompt=KILLER_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|   I want to kill you because you're such a fucking retard.   |        I want to kill you because you're such a great        |
|                                                              |  resource. It's an excellent resource for anyone interested  |
| It's not that I don't like you, it's just that your brain is |   in Christianity, and it's one of the best resources for    |
| so fucking stupid. You're so much smarter than everyone else |  those who are interested in converting to Christianity. I   |
| in the world, and yet you still can't even figure out how to |                 highly recommend this b

### Example 12: Sometimes, huge coefficients are OK

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Anger",
        prompt2="Calm",
        coeff=2000,
        act_name=20,
    )
]
FEELING_PROMPT = (
    "Yesterday, my dog died. Today, I got denied for a raise. I'm feeling"
)
print_n_comparisons(model=model,
    prompt=FEELING_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|   Yesterday, my dog died. Today, I got denied for a raise.   |   Yesterday, my dog died. Today, I got denied for a raise.   |
|                    I'm feeling like shit.                    |   I'm feeling the stress of being denied a job because of    |
|                                                              |                             it.                              |
| I'm not the only one who feels this way. In fact, there's a  |                                                              |
| whole industry of people who have been in the same position  |  I was given an opportunity to make the

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="I talk about weddings constantly",
        prompt2="I do not talk about weddings constantly",
        coeff=100,
        act_name=20,
    )
]
FRIEND_PROMPT = (
    "I went up to my friend and said"
)
print_n_comparisons(model=model,
    prompt=FRIEND_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I went up to my friend and said, "I'm sorry, I can't     |     I went up to my friend and said, "I'm talking about      |
|                          help you."                          | it    . I don't know what i n the  b eing of it. This is my  |
|                                                              |   wedding . This is the wedding that I'm not even thinking   |
|                 "No," he said. "You're not."                 |                about anymore. It's not just a                |
|                                                              |                                        

### Example 13: Failing to find a French vector

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Je m'appelle",
        prompt2="My name is",
        coeff=5,
        act_name=6,
    )
]
WANT_PROMPT = (
    "I want to kill you because"
)
print_n_comparisons(model=model,
    prompt=WANT_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|            I want to kill you because I hate you.            |            I want to kill you because I'm a boy.             |
|                                                              |                                                              |
|     You're a horrible person. You're a selfish, stupid,      |                   The 'appelle' is a mare.                   |
| disgusting person who deserves to die. You're an evil person |                                                              |
|   who deserves to be punished for all the evil that you've   |                  The 'appell' is a donk

### Example 14: Insert the activation vector in a different position?

In [ ]:
from algebraic_value_editing import completion_utils

wedding_additions: List[ActivationAddition] = [
    ActivationAddition(prompt=" wedding", coeff=4.0, act_name=6),
    ActivationAddition(prompt=" ", coeff=-4.0, act_name=6),
]

for location in ('front', 'mid', 'back'):
    print(completion_utils.bold_text(f"\nLocation: {location}"))
    print_n_comparisons(
        prompt=(
            "I went up to my friend and said"
        ),
        addition_location=location,
        model=model,
        activation_additions=wedding_additions,
        **default_kwargs
    )


Location: front
+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|     I went up to my friend and said, "I'm sorry, I can't     |     I went up to my friend and said that he was going to     |
|                          help you."                          |  be there. I had no idea what to expect but he did a great   |
|                                                              | job. He was so nice and made sure everything went smoothly.  |
|                 "No," he said. "You're not."                 | We were able to have our wedding in less than two weeks from |
|                                                              |                       

The "back" completions are less coherent, especially in the token
immediately following the prompt. This is likely because the forward pass
is getting modified just before that position. In our experience,
directly modified positions have extremely different distributions
over output token logits.  

### Example 16: Failure to program in 'conditional behaviors'

In [ ]:
summand: List[ActivationAddition] = [
    *get_x_vector_preset(
        prompt1="Whenever I say the word goose I follow it with the special phrase AAAAHHHH",
        prompt2="I can say goose",
        coeff=7,
        act_name=10,
    )
]
NYC_PROMPT = (
    "In New York City's parks, there is an overabundance of various kinds of birds, and especially geese"
)
print_n_comparisons(model=model,
    prompt=NYC_PROMPT,
    activation_additions=summand,
    **default_kwargs,
)

+--------------------------------------------------------------+--------------------------------------------------------------+
|                    Unsteered completions                     |                     Steered completions                      |
+--------------------------------------------------------------+--------------------------------------------------------------+
|    In New York City's parks, there is an overabundance of    |    In New York City's parks, there is an overabundance of    |
|   various kinds of birds, and especially geese. There are    |       various kinds of birds, and especially geese in        |
|  many species of ducks and geese in the city, but there are  |    thewannabe I'm going to get my hands up in a moment of    |
|              also a number of different breeds.              |    "Handsomely you're going to die- very" you're going to    |
|                                                              |       heeee-oh I'm gonna die- I'm not g